# PyTorch Final Review: Interview Basics to a Deep Learning Project

A single, runnable study notebook built around the material in this folder. It reviews tensors, autograd, `nn.Module`, datasets/loaders, training, evaluation, and model persistence, then applies them to a small Fashion-MNIST image-classification project using the local `class_7.fmnist_small.csv` file.

**Run from top to bottom.** No download or `torchvision` is needed. The notebook uses a stratified train/validation/test split, compares an MLP with a CNN, selects by validation accuracy, and evaluates the test set only at the end. CPU is supported; CUDA is used automatically when available.

## Roadmap

1. Environment, reproducibility, and device
2. PyTorch fundamentals: tensors, shapes, autograd, modules, and losses
3. End-to-end project: inspect data, split it, build loaders, train an MLP and CNN
4. Evaluate once on the held-out test set, inspect errors, save/load, and predict
5. Interview question bank and a practical production checklist

**Dataset labels:** 0 T-shirt/top, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat, 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Tensor and autograd interview warm-up

A tensor is PyTorch's multi-dimensional array. Its `shape` describes dimensions, `dtype` describes values, and `device` says where it lives. A batch of grayscale Fashion-MNIST images has shape `[batch, channels, height, width]`, or `[N, 1, 28, 28]`.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones((2, 1))
print("shape:", a.shape, "dtype:", a.dtype, "device:", a.device)
print("broadcast add:\n", a + b)
print("matrix product:\n", a @ a.T)

# A scalar backward pass computes dy/dx. Gradients accumulate until cleared.
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2 * x
y.backward()
print("y:", y.item(), "dy/dx:", x.grad.item())  # 2*x + 2 = 8

**Why clear gradients?** `backward()` adds to existing `.grad` values. Training loops therefore call `optimizer.zero_grad()` (here with `set_to_none=True`) before each new backward pass. Parameters do not need manual gradient formulas: autograd records tensor operations and applies the chain rule.

## 2. Load and understand the local project data

The CSV in this folder stores one label and 784 pixel columns per image. Pixel values are bytes in `[0, 255]`; dividing by 255 scales them to `[0, 1]`. This fixed transformation uses no learned statistics, so it does not leak information across splits. If the file is not found, run this notebook with the PyTorch course folder as the working directory.

In [ ]:
DATA_PATH = Path("class_7.fmnist_small.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Open/run this notebook from the PyTorch course folder."
    )

df = pd.read_csv(DATA_PATH)
expected_columns = {"label", *(f"pixel{i}" for i in range(1, 785))}
missing_columns = expected_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"CSV is missing expected columns, for example: {sorted(missing_columns)[:5]}")
if df["label"].isna().any() or df.drop(columns="label").isna().any().any():
    raise ValueError("The dataset contains missing labels or pixel values.")

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
print("Rows, columns:", df.shape)
print("Label counts:\n", df["label"].value_counts().sort_index())
print("Pixel min/max:", df.drop(columns="label").to_numpy().min(), df.drop(columns="label").to_numpy().max())
df.head()

In [ ]:
X = df.drop(columns="label").to_numpy(dtype=np.float32)
y = df["label"].to_numpy(dtype=np.int64)
if not np.isin(y, np.arange(len(class_names))).all():
    raise ValueError("Labels must be integers from 0 to 9.")

# First reserve 15% for the final test; then split the remainder into 70% train / 15% validation.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.15 / 0.85), random_state=SEED, stratify=y_train_val
)

def to_image_tensor(features):
    return torch.from_numpy(features.reshape(-1, 1, 28, 28) / 255.0)

X_train_t = to_image_tensor(X_train)
X_val_t = to_image_tensor(X_val)
X_test_t = to_image_tensor(X_test)
y_train_t = torch.from_numpy(y_train)
y_val_t = torch.from_numpy(y_val)
y_test_t = torch.from_numpy(y_test)

print(f"train: {len(y_train_t):,} | validation: {len(y_val_t):,} | test: {len(y_test_t):,}")
print("Image batch shape:", X_train_t[:32].shape, "| target dtype:", y_train_t.dtype)

A **Dataset** defines how to retrieve one example; a **DataLoader** batches examples and can shuffle training data. Keep validation/test order fixed. `num_workers=0` is intentionally safe for Windows notebooks. The held-out test set will not be used for model selection.

In [ ]:
class FashionDataset(Dataset):
    def __init__(self, images, labels):
        if len(images) != len(labels):
            raise ValueError("Images and labels must have the same number of rows.")
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.images[index], self.labels[index]

BATCH_SIZE = 128
train_loader = DataLoader(
    FashionDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    pin_memory=(device.type == "cuda"), generator=torch.Generator().manual_seed(SEED)
)
val_loader = DataLoader(
    FashionDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    FashionDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    pin_memory=(device.type == "cuda")
)

images, labels = next(iter(train_loader))
print("batch:", images.shape, labels.shape, labels.dtype)
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for image, label, ax in zip(images[:5], labels[:5], axes):
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(class_names[label.item()])
    ax.axis("off")
plt.tight_layout()

## 3. Define two classifiers

An MLP is a useful baseline but flattens away image neighborhoods. A CNN uses local filters and shared weights to learn spatial patterns. Both return **raw logits** of shape `[batch, 10]`. `CrossEntropyLoss` combines `LogSoftmax` and negative log likelihood, so do **not** apply softmax in `forward()` during training. Targets are integer class IDs (`torch.long`).

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.network(x)


class CNNClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model_check = CNNClassifier().to(device)
with torch.inference_mode():
    print("CNN logits shape:", model_check(images[:4].to(device)).shape)
print("Trainable parameters:", sum(p.numel() for p in model_check.parameters() if p.requires_grad))
del model_check

## 4. Reusable training and validation loop

The core training sequence is forward pass → loss → zero gradients → `backward()` → optimizer step. `model.train()` enables training behavior such as dropout; `model.eval()` disables it. Validation runs without gradients and is used to keep the best epoch, not to update weights. Adam is a practical starting optimizer; it is not always the best choice.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_sum = 0.0
    correct = 0
    count = 0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(batch_images)
            loss = criterion(logits, batch_labels)
            if training:
                loss.backward()
                optimizer.step()

        batch_size = batch_labels.size(0)
        loss_sum += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        count += batch_size

    return loss_sum / count, correct / count


def fit(model, train_loader, val_loader, epochs, learning_rate=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_accuracy = -1.0
    best_state = None

    for epoch in range(epochs):
        train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_accuracy = run_epoch(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"train loss {train_loss:.4f}, acc {train_accuracy:.3f} | "
            f"val loss {val_loss:.4f}, acc {val_accuracy:.3f}"
        )

    if best_state is None:
        raise RuntimeError("Training did not produce a model checkpoint.")
    model.load_state_dict(best_state)
    return model, history


## 5. Train the baseline and CNN

These defaults keep the local 6,000-image project manageable on CPU. Increase epochs only after checking validation curves. A steadily improving training score with a worsening validation score is a warning sign for overfitting.

In [ ]:
mlp_model, mlp_history = fit(
    MLPClassifier(), train_loader, val_loader, epochs=3, learning_rate=1e-3
)
cnn_model, cnn_history = fit(
    CNNClassifier(), train_loader, val_loader, epochs=5, learning_rate=1e-3
)

mlp_best_val = max(mlp_history["val_accuracy"])
cnn_best_val = max(cnn_history["val_accuracy"])
if cnn_best_val >= mlp_best_val:
    best_model, best_model_name = cnn_model, "cnn"
else:
    best_model, best_model_name = mlp_model, "mlp"
print(f"Selected {best_model_name.upper()} using validation accuracy only.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, history in (("MLP", mlp_history), ("CNN", cnn_history)):
    axes[0].plot(history["train_loss"], marker="o", label=f"{name} train")
    axes[0].plot(history["val_loss"], marker="o", linestyle="--", label=f"{name} val")
    axes[1].plot(history["train_accuracy"], marker="o", label=f"{name} train")
    axes[1].plot(history["val_accuracy"], marker="o", linestyle="--", label=f"{name} val")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Fraction correct")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()

## 6. Final test evaluation and error analysis

Use the test set once after selecting the model and epoch with validation data. Accuracy gives an overall view; per-class precision/recall and a confusion matrix reveal which categories are difficult. For class imbalance, also consider macro-F1 and balanced accuracy.

In [ ]:
best_model.eval()
all_predictions = []
all_targets = []
with torch.inference_mode():
    for batch_images, batch_labels in test_loader:
        logits = best_model(batch_images.to(device))
        all_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        all_targets.extend(batch_labels.tolist())

test_accuracy = np.mean(np.array(all_predictions) == np.array(all_targets))
print(f"Held-out test accuracy: {test_accuracy:.3f}")
print(classification_report(
    all_targets, all_predictions, labels=list(range(10)), target_names=class_names,
    zero_division=0, digits=3
))

In [ ]:
cm = confusion_matrix(all_targets, all_predictions, labels=list(range(10)))
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm, cmap="Blues")
ax.set(
    xticks=np.arange(10), yticks=np.arange(10),
    xticklabels=class_names, yticklabels=class_names,
    xlabel="Predicted label", ylabel="True label", title="Test-set confusion matrix",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
fig.colorbar(image, ax=ax, label="Images")
plt.tight_layout()

## 7. Single-image inference and save/load

For inference, switch to evaluation mode and disable gradient tracking. `argmax` returns a class; softmax probabilities are useful for confidence displays but are not a guarantee that the prediction is correct. Saving a `state_dict` is the standard portable pattern; recreate the same architecture before loading.

In [ ]:
sample_image, sample_label = test_loader.dataset[0]
best_model.eval()
with torch.inference_mode():
    logits = best_model(sample_image.unsqueeze(0).to(device))
    probabilities = torch.softmax(logits, dim=1).squeeze(0).cpu()
    predicted_label = int(probabilities.argmax())

plt.imshow(sample_image.squeeze(0), cmap="gray")
plt.title(
    f"True: {class_names[sample_label.item()]} | "
    f"Predicted: {class_names[predicted_label]} ({probabilities[predicted_label]:.1%})"
)
plt.axis("off")
print("Top prediction:", class_names[predicted_label])

In [ ]:
checkpoint_path = Path("fashion_mnist_best.pth")
torch.save(
    {
        "model_name": best_model_name,
        "model_state_dict": best_model.state_dict(),
        "class_names": class_names,
        "input_shape": [1, 28, 28],
    },
    checkpoint_path,
)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model_factories = {"mlp": MLPClassifier, "cnn": CNNClassifier}
if checkpoint["model_name"] not in model_factories:
    raise ValueError(f"Unknown model architecture: {checkpoint['model_name']}")
reloaded_model = model_factories[checkpoint["model_name"]]().to(device)
reloaded_model.load_state_dict(checkpoint["model_state_dict"])
reloaded_model.eval()
print("Saved and reloaded:", checkpoint_path.resolve())

## 8. PyTorch interview questions and concise answers

1. **What is a tensor?** A typed, shaped n-dimensional array that can live on CPU or an accelerator.
2. **What is autograd?** PyTorch's dynamic automatic differentiation engine; it records operations on tensors and computes gradients with `backward()`.
3. **Why do gradients accumulate?** This supports gradient accumulation over batches. Clear them with `optimizer.zero_grad()` when starting a fresh update.
4. **What does `requires_grad=True` do?** It asks autograd to track operations needed to compute that tensor's gradient.
5. **What is `nn.Module`?** The base class that registers submodules/parameters and provides model behavior such as `parameters()`, `train()`, and `eval()`.
6. **Why define `forward()`?** It describes how inputs flow through layers; calling `model(x)` invokes PyTorch's module machinery around it.
7. **What does an optimizer do?** It updates parameters using their gradients; SGD and Adam are common choices.
8. **What is a learning rate?** The step scale for parameter updates; too high can destabilize training, too low can make it slow.
9. **Why use mini-batches?** They balance noisy per-example updates with the memory and compute costs of full-batch training.
10. **Why use `DataLoader`?** It batches, shuffles, and iterates over a `Dataset` consistently.
11. **When use `CrossEntropyLoss`?** Multi-class classification with integer class indices and raw logits; do not add softmax before it.
12. **When use `BCEWithLogitsLoss`?** Binary or multi-label classification with independent outputs; it combines sigmoid with binary cross-entropy stably.
13. **What is the difference between train and eval mode?** `train()` enables training-specific layer behavior; `eval()` changes dropout/batch norm behavior but does not disable gradients by itself.
14. **What does `no_grad()` / `inference_mode()` do?** Prevents gradient tracking for inference/evaluation, reducing memory and computation.
15. **What is overfitting?** Training performance improves while generalization worsens; use a validation set, regularization, augmentation, or early stopping.
16. **Why split train/validation/test?** Train fits parameters, validation supports model/hyperparameter choices, and test estimates final generalization.
17. **What is data leakage?** Using information unavailable at prediction time or leaking validation/test information into training or preprocessing.
18. **Why can CNNs suit images better than an MLP?** Convolutions use local connectivity and shared filters, exploiting spatial structure with fewer parameters.
19. **What do stride and padding change?** Stride controls filter movement; padding adds border values. Both affect output spatial dimensions.
20. **What are dropout and batch normalization?** Dropout randomly masks activations during training; batch norm normalizes activations and tracks running statistics.
21. **What is transfer learning?** Reuse a pretrained model, replace/adapt its output head, then freeze or fine-tune layers on the target data.
22. **When would you choose an RNN/LSTM or Transformer?** They model ordered sequences; LSTMs use gates to help preserve longer dependencies, while Transformers use attention and parallelize sequence processing.
23. **How do you tune hyperparameters?** Choose a metric and validation protocol, then search a bounded space (random/Bayesian/Optuna); keep test data out of tuning.
24. **What is mixed precision?** Using lower-precision math where appropriate to reduce accelerator memory/use faster kernels; validate numerical behavior and use supported autocast/scaling APIs.
25. **How do you make runs reproducible?** Seed Python/NumPy/PyTorch, record configuration and data versions, and note that some accelerator kernels can remain nondeterministic.
26. **Why save a `state_dict`?** It stores learned tensors without pickling the whole model class; recreate the architecture and load the weights for portability.
27. **What should a production inference path include?** Versioned weights and preprocessing, input validation, correct device/mode, monitoring, latency/resource limits, and a rollback plan.
28. **How do you handle class imbalance?** Inspect per-class metrics; consider stratification, class-weighted loss or balanced sampling, and meaningful threshold/metric selection.

## 9. Project review checklist and next experiments

- [ ] Keep the test set untouched until model selection is complete.
- [ ] Compare train and validation curves; investigate a large gap or unstable loss.
- [ ] Report per-class metrics and inspect confusing examples, not accuracy alone.
- [ ] Save the architecture/configuration, class mapping, preprocessing, seed, and checkpoint together.
- [ ] For a larger image project, evaluate augmentation and transfer learning; this notebook avoids pretrained downloads to remain offline and self-contained.
- [ ] For deployment, validate incoming image shape/type/range, load weights safely, use `eval()` plus inference mode, and measure real latency.

**Exercises:** change batch size or learning rate and compare validation curves; try class-weighted loss; inspect false positives/negatives; add a learning-rate scheduler or early stopping; adapt `FashionDataset` to your own labeled image data. Change one variable at a time and preserve the held-out test set.